# MCP 로 남이 만든 도구 붙이기

이 노트북은 **읽는 자료**입니다. 실행할 코드는 없습니다. 개념을 먼저 잡고,
손으로 하는 실습은 같은 폴더의 **`01_파일시스템_MCP` ~ `06_종합_데이터분석_자동화`** 에서 합니다.
같은 이름의 노트북(`.ipynb`)과 스크립트(`.py`)가 함께 있으니 편한 쪽으로 실습하세요(차이는 아래에 정리해 두었습니다).

## ⏪ 복습

- **`@tool`**: 파이썬 함수 위에 붙이면 에이전트가 쓸 수 있는 **도구**가 됩니다. docstring 이 곧 사용설명서입니다.
- **`create_agent(model, tools, system_prompt)`**: 모델에 도구 목록을 붙여 에이전트를 만듭니다.
- **도구 설계 원칙**: 하나의 일만, 결과는 문자열로, 에러도 문자열로.

지금까지 도구는 **전부 우리가 직접 만들었습니다.** 오늘은 반대편을 봅니다. **남이 이미 만들어 둔 도구**를 가져다 씁니다.

**오늘의 목표**

- [ ] 도구를 매번 직접 만드는 방식의 **한계**를 설명할 수 있다.
- [ ] **MCP** 가 무엇을 표준화하는지, 호스트·클라이언트·서버가 각각 무엇인지 안다.
- [ ] 서버에 붙는 두 가지 방식(**stdio · HTTP**)을 구분하고 언제 무엇을 쓸지 고른다.
- [ ] 우리가 만든 `@tool` 과 **MCP 도구의 차이**(비동기·반환값)를 안다.


---
# 1. MCP 는 왜 필요한가

지금까지 도구는 전부 우리가 만들었습니다.

```python
@tool
def branch_ranking(top: int) -> str:
    """지점별 매출 상위 top 개를 돌려준다."""
    ...
```

우리 데이터·우리 규칙이니 우리가 만드는 게 맞습니다. 그런데 **웹 검색 · 파일 읽기 · 코드 실행 · 문서 조회**처럼
누구에게나 똑같은 기능까지 매번 만들어야 할까요? 검색 하나만 해도 API 계약·요청 형식·결과 파싱·에러 처리·
요금 제한을 전부 떠안아야 하고, 그렇게 만든 도구는 **내 프로젝트 안에만** 남습니다.

## 조합 폭발

앱 3개와 기능 4개만 있어도 **12벌**을 만들고 관리해야 합니다.

<img src="../images/tool_duplication.png" width="820">

**MCP** 는 이 가운데에 **규격**을 하나 끼워 넣어 만드는 쪽과 쓰는 쪽을 떼어 놓습니다.
그러면 **도구는 기능마다 한 벌씩, 4벌만** 만들면 됩니다.
앱 쪽이 하는 일은 도구를 만드는 게 아니라 **MCP 클라이언트를 한 번 붙이는 일**이고, 그것도 대개 라이브러리가 대신해 줍니다
(이 단원에서 쓸 설정 딕셔너리 몇 줄이 그것입니다).
손댈 곳을 앱까지 다 세어도 **4 + 3 = 7**. 핵심은 숫자가 아니라 **곱셈(3 × 4)이 덧셈(3 + 4)으로 바뀐다**는 것입니다.

<img src="../images/mcp_standard.png" width="820">

---
# 2. MCP 는 무엇인가

> **MCP(Model Context Protocol)** 는 **AI 애플리케이션과 도구를 들고 있는 프로그램이 주고받는 메시지의 형식을 정해 둔 공개 규약**입니다.
>
> 규약에 적힌 것은 세 가지입니다. **어떤 도구가 있는지 묻는 방법**, **그 도구를 호출하는 방법**, **결과를 돌려주는 형식**.
>
> 그래서 MCP 는 모델도, 프레임워크도, 도구 그 자체도 아닙니다. 도구를 **주고받는 방법의 약속**이라, 앱 만드는 쪽과 도구 만드는 쪽이 서로 모른 채 따로 만들어도 말이 통합니다.


---
# 3. 등장인물 3명: 호스트 · 클라이언트 · 서버

<img src="../images/mcp_three_roles.png" width="820">

| 이름 | 무엇 | 이 수업에서는 |
|---|---|---|
| **호스트(Host)** | 모델을 부르는 AI 애플리케이션 | 우리가 쓰는 실습 파일(에이전트) |
| **클라이언트(Client)** | 서버와 규약대로 통신하는 부품 | 라이브러리(`langchain_mcp_adapters`)의 `MultiServerMCPClient` 를 불러 씁니다 |
| **서버(Server)** | 도구·데이터를 실제로 들고 있는 프로그램 | 남이 만들어 공개한 서버들(검색·파일·코드실행·문서조회) |

> **이 단원은 MCP 서버를 만들지 않습니다. 이미 만들어져 공개된 서버를 가져다 씁니다.**

---
# 4. 서버가 내주는 3가지

| 종류 | 무엇 | 예 | 이 단원 |
|---|---|---|---|
| **도구(Tools)** | 모델이 **호출**해서 뭔가를 하게 하는 함수 | `search(query)` · `run-code(code)` | ✅ 중심 |
| **리소스(Resources)** | 모델이 **읽을** 데이터. 읽기 전용 파일·문서 같은 것 | 파일 내용, DB 스키마 | 참고만 |
| **프롬프트(Prompts)** | 서버가 미리 준비해 둔 **프롬프트 템플릿** | "이 코드를 리뷰해 줘" 틀 | 참고만 |

우리는 **도구**만 씁니다. 우리가 `@tool` 로 만든 것과 **똑같은 자리**에 들어가기 때문입니다. 
`create_agent(model, [우리 도구 + MCP 도구])` 처럼 한 리스트에 섞어 넘길 수 있습니다.

---
# 5. 붙는 방법 두 가지: stdio 와 HTTP

연결 설정은 **딕셔너리 하나**입니다. 전송 방식에 따라 모양이 둘로 갈립니다.

```python
# 1) stdio -- 서버를 내 컴퓨터에서 프로세스로 띄우고, 표준입출력으로 대화한다
FILESYSTEM = {"command": "npx",
              "args": ["-y", "@modelcontextprotocol/server-filesystem", "/실습/day21"],
              "transport": "stdio"}

WEB_SEARCH = {"command": "uvx",
              "args": ["duckduckgo-mcp-server"],
              "transport": "stdio"}

# 2) streamable_http -- 이미 떠 있는 원격 서버에 주소로 붙는다
CONTEXT7 = {"url": "https://mcp.context7.com/mcp", "transport": "streamable_http"}
```

## 키의 뜻

| 키 | 뜻 | 예 |
|---|---|---|
| `command` | 서버를 띄울 **실행 파일** 이름 | `npx` · `uvx` |
| `args` | 그 실행 파일에 넘길 **명령줄 인자** 리스트 | `["-y", "@modelcontextprotocol/server-filesystem", "/실습/day21"]` |
| `url` | 원격 서버의 **주소**(stdio 에는 없습니다) | `"https://mcp.context7.com/mcp"` |
| `transport` | 서버와 말을 주고받는 **통로** | `"stdio"` · `"streamable_http"` |

`command` 와 `args` 는 어려운 개념이 아닙니다. **터미널에 칠 한 줄을 쪼개 적은 것**입니다.

```bash
npx -y @modelcontextprotocol/server-filesystem /실습/day21
└─┬─┘ └──────────────────────┬────────────────────────┘
command                     args
```

## `npx` 와 `uvx`: 설치 없이 패키지를 실행하는 도구

MCP 서버는 남이 배포해 둔 **패키지**입니다. 그 패키지를 미리 설치하지 않고 **그때그때 내려받아 바로 실행**해 주는 실행기가 둘 있습니다.

| 실행기 | 무엇을 실행하나 | 어디서 딸려 오나 | 자주 붙는 인자 |
|---|---|---|---|
| `npx` | Node.js(npm) 패키지 | Node.js 를 깔면 함께 | `-y`: 설치할지 묻지 않고 진행 |
| `uvx` | 파이썬(PyPI) 패키지 | `uv` 를 깔면 함께 | 없음(패키지 이름부터) |

- `npx -y @modelcontextprotocol/server-filesystem /실습/day21` : 파일시스템 서버. **맨 뒤 경로가 서버가 볼 수 있는 폴더**이고, 그 밖은 건드리지 못합니다.
- `uvx duckduckgo-mcp-server` : 검색 서버. 파이썬으로 만들어진 서버라 `uvx` 로 띄웁니다.

> 첫 실행이 수십 초 걸리는 이유가 여기 있습니다. 패키지를 내려받는 시간입니다. 두 번째부터는 캐시에 있어 빠릅니다.

> **`transport` 는 서버와 어느 통로로 말할지 고르는 값입니다.** 내 컴퓨터에 띄운 프로세스와 표준입출력으로 주고받으면 `"stdio"`, 이미 떠 있는 원격 서버와 HTTP 로 주고받으면 `"streamable_http"` 입니다.
>
> 통로가 정해지면 함께 적을 키도 정해집니다. stdio 는 `command`·`args`, HTTP 는 `url` 입니다. 셋 다 **생략할 수 없습니다.**

| | **stdio** | **streamable_http** |
|---|---|---|
| 서버가 도는 곳 | 내 컴퓨터(프로세스) | 남의 서버(원격) |
| 필요한 것 | 실행기(`npx`·`uvx`)와 패키지 | 주소(URL) |
| 내 파일 접근 | 가능 | 불가(원격이라 당연) |
| 인터넷 | 설치할 때만 | 항상 필요 |
| 흔한 쓰임 | 파일·로컬 코드 실행 | 회사·서비스가 공개한 공용 서버 |

> **어느 쪽을 고를까**: 내 컴퓨터의 파일·프로그램을 다뤄야 하면 **stdio**,
> 남이 운영하는 서비스를 쓰는 것이면 **HTTP** 입니다.

---
# 6. 우리가 만든 도구 vs MCP 도구

붙이고 나면 **똑같은 LangChain 도구**입니다. 하지만 쓸 때 다른 점이 셋 있습니다.

| | 우리 `@tool` | MCP 도구 |
|---|---|---|
| 만든 사람 | 우리 | 남(공개 서버) |
| 실행되는 곳 | 우리 프로세스 안 | **서버 프로세스/원격** |
| 호출 | `tool.invoke({...})` 도 되고 `ainvoke` 도 됨 | **`await tool.ainvoke({...})` 만 됨** |
| 반환값 | 우리가 정한 문자열 | **콘텐츠 블록 리스트** `[{'type': 'text', 'text': ...}]` |

세 번째·네 번째가 실제로 학생이 걸리는 자리입니다.

```python
# 동기 호출은 실패한다 -- NotImplementedError: StructuredTool does not support sync invocation.
tool.invoke({"query": "MCP"})

# 비동기 호출이라야 한다
result = await tool.ainvoke({"query": "MCP"})

# 반환값은 문자열이 아니라 콘텐츠 블록 리스트다 -> result[0][\"text\"] 로 꺼낸다
[{'type': 'text', 'text': '검색 결과 3건 ...', 'id': 'lc_...'}]
```

> 도구 중 **하나라도 MCP 도구면 에이전트도 `await agent.ainvoke(...)`** 로 불러야 합니다.
> `agent.invoke(...)` 는 같은 이유로 실패합니다.

**그래서 실습은 처음부터 끝까지 비동기로 돌아갑니다.** 노트북(`.ipynb`)에서는 셀에서 바로 `await` 를 쓰고,
스크립트(`.py`)에서는 `async def main()` 안에 코드를 담고 파일 맨 아래 `asyncio.run(main())` 한 줄로 실행합니다.

```python
client = MultiServerMCPClient({"search": WEB_SEARCH})   # 연결 설정에 별명을 붙여 클라이언트를 만든다
tools = await client.get_tools()                        # 서버에 붙어 도구 목록을 받는다
by_name = {tool.name: tool for tool in tools}           # 이름으로 꺼내 쓰려고 딕셔너리로

result = await by_name["search"].ainvoke({"query": "MCP", "max_results": 3})
print(result[0]["text"])   # 사람이 읽을 값은 첫 블록의 text
pprint(result)             # 구조를 그대로 보고 싶을 때
```

브라우저처럼 **상태를 가진 서버**(열어 둔 페이지가 다음 호출까지 남아야 하는 서버)는 세션을 열어 둔 채 부릅니다.

```python
async with client.session("web") as session:      # 이 블록 동안 서버가 살아 있다
    tools = await load_mcp_tools(session)
    await by_name["browser_navigate"].ainvoke({"url": "https://example.com"})
    await by_name["browser_snapshot"].ainvoke({})  # 앞에서 연 페이지를 그대로 읽는다
```

---
# 7. 이 단원에서 붙여 볼 서버

전부 **API 키 없이** 쓸 수 있는 공개 서버입니다.

| 서버 | 무엇을 해 주나 | 전송 | 실습 파일 |
|---|---|---|---|
| **파일시스템** `@modelcontextprotocol/server-filesystem` | 정해 준 폴더 안의 파일 목록·읽기·쓰기 | stdio(npx) | `01_파일시스템_MCP` |
| **웹 검색** `duckduckgo-mcp-server` | 웹 검색과 본문 가져오기 | stdio(uvx) | `02_웹검색_MCP` |
| **브라우저 조작** `@playwright/mcp` | 페이지를 열고 화면 구조를 읽고 클릭·입력 | stdio(npx) | `03_브라우저조작_Playwright` |
| **DB 조회** `mcp-server-sqlite` | DB 의 표 목록·열 구성 확인과 `SELECT` 실행 | stdio(uvx) | `04_SQLite_MCP` |
| **코드 실행** `mcp-server-code-runner` | 받은 코드를 실행하고 출력을 돌려줌 | stdio(npx) | `05_코드실행_MCP` |

마지막 `06_종합_데이터분석_자동화` 에서는 DB·코드 실행·파일시스템 서버 셋을 한꺼번에 붙입니다.

원격(HTTP) 서버는 **과제에서 직접 붙어 봅니다.** 라이브러리 문서를 찾아 주는 Context7(`https://mcp.context7.com/mcp`)이고,
설정은 주소 한 줄뿐입니다.

그 밖에 실무에서 자주 보이는 서버들(참고): GitHub·Slack·Notion·Google Drive·Postgres.
대부분 **접근 토큰**이 필요해서 이 수업에서는 다루지 않습니다.

---
# 8. 노트북(`.ipynb`)과 스크립트(`.py`), 어느 쪽으로 실습하나

`교안_01_MCP_클라이언트/` 폴더에는 실습 파일이 **같은 이름으로 두 벌** 있습니다. 내용은 같고, 노트북 판에는 설명과 따라하기가 더해져 있습니다.

- **맥·리눅스**: 어느 쪽이든 됩니다. 주피터 커널이 stdio MCP 서버를 자식 프로세스로 띄울 수 있습니다.
- **윈도우**: `.py` 판으로 하세요. MCP 의 `stdio` 방식은 서버를 **자식 프로세스로 띄우는데**, 윈도우의 주피터 커널이 쓰는 이벤트 루프는 그것을 지원하지 않아 그 자리에서 실패할 수 있습니다.

노트북 판은 `.py` 판과 코드가 세 곳 다릅니다.

| `.py` | 노트북 | 왜 |
|---|---|---|
| `asyncio.run(main())` | 셀에서 바로 `await` | 주피터는 이미 이벤트 루프를 돌리고 있어 `asyncio.run` 을 쓰면 `RuntimeError` 가 난다 |
| `Path(__file__).parent` | `Path.cwd()` | 노트북에는 `__file__` 이 없다 |
| `async with client.session(...)` | `await client.get_tools(...)` | `async with` 블록은 셀이 끝나면 닫혀 다음 셀에서 도구가 죽는다(03 번만 예외로 세션을 직접 열고 닫는다) |

`async`·`await` 가 처음이면 일차 폴더의 **`부록_동기_비동기_기초.ipynb`** 를 먼저 열어 보세요.

---
## 이번 강의 정리

- 도구를 앱마다 새로 만들면 **같은 코드가 앱 수만큼 복제**된다 -> MCP 는 그 사이에 **규격**을 끼워 넣는다.
- **호스트(우리 앱) · 클라이언트(`MultiServerMCPClient`) · 서버(남이 만든 도구 제공자)** 세 명이 등장한다.
- 붙는 방법은 **stdio**(내 컴퓨터에서 프로세스로) 와 **HTTP**(원격 주소로) 두 가지다.
- MCP 도구는 **비동기 전용**이고 반환값이 **콘텐츠 블록 리스트**다.